# Теория. Веб-приложение на FastAPI: маршруты, модели Pydantic, автоматическая документация
## «Сколько кода нашего сервера из Т4.Б1.1 было про сервис «Покрытие», а сколько про сам протокол HTTP?»

**Цель:** перенести сервер из Т4.Б1.1 на фреймворк FastAPI и понять, какую работу фреймворк берет на себя. После занятия вы объявляете маршруты декораторами, получаете параметры пути и запроса с проверкой типов, описываете входные и выходные данные моделями Pydantic и читаете автоматическую документацию программного интерфейса на странице `/docs`.

**Как работать с ноутбуком.** Ячейки с кодом выполняются по порядку (Shift+Enter). Запускать сервер для них не нужно: примеры отправляют запросы через тестовый клиент FastAPI `TestClient`, который передает запрос приложению прямо в памяти. Тот же клиент используют тесты практической работы.

**План:**
1. Что делает фреймворк вместо нас
2. Первое приложение и запуск командой `fastapi dev`
3. Маршруты и методы, параметры пути и запроса
4. Проверка параметров и ответ 422
5. Модели Pydantic
6. Тело запроса и код 201
7. Модель ответа
8. Автоматическая документация
9. Функции `def` и `async def`
10. Итоги, словарь и вопросы

## 1. Что делает фреймворк вместо нас

Вспомним сервер из Т4.Б1.1. Чтобы ответить на запрос `GET /braking?v=60&mu=0.7`, мы написали девять функций:

| Работа | Функция в Т4.Б1.1 | Кто делает в FastAPI |
| --- | --- | --- |
| принять соединение, прочитать байты до конца тела | `serve`, `read_request` | сервер Uvicorn |
| разобрать строку запроса, заголовки, параметры | `parse_request`, `parse_query` | FastAPI (через библиотеку Starlette) |
| выбрать функцию по методу и пути, ответить 404 и 405 | `route`, `method_not_allowed` | FastAPI по декораторам `@app.get`, `@app.post` |
| превратить строку `"60"` в число и проверить диапазон | `read_number` | FastAPI по подсказкам типов (через библиотеку Pydantic) |
| собрать ответ, посчитать `Content-Length` в байтах | `build_response` | FastAPI |
| отдать файлы с правильным `Content-Type` | `static_response` | `StaticFiles` |

**Фреймворк** – библиотека, которая задает каркас программы: мы пишем только свои части (что ответить на запрос), а фреймворк сам вызывает их в нужный момент. Обычная библиотека, наоборот, ждет, когда ее вызовем мы.

FastAPI собран из трех частей:
- **Uvicorn** – сервер: принимает соединения TCP и разбирает протокол HTTP. Он работает по стандарту **ASGI** (Asynchronous Server Gateway Interface, асинхронный интерфейс между веб-сервером и приложением на Python). Благодаря стандарту сервер и приложение можно менять независимо;
- **Starlette** – маршрутизация, запросы и ответы, файлы, промежуточные обработчики;
- **Pydantic** – проверка и преобразование данных по подсказкам типов.

**Сравнение с Flask.** Flask вы частично знаете. Маршруты в нем объявляются похоже:

```python
# Flask                                   # FastAPI
@app.get("/api/surfaces")                 @app.get("/api/surfaces")
def list_surfaces():                      def list_surfaces():
    return [...]                              return [...]
```

Разница видна, когда появляются данные запроса. Во Flask параметры достаются из `request.args` строками и проверяются вручную, а документация не строится. В FastAPI параметры – аргументы функции с подсказками типов: по ним фреймворк сам проверяет данные и строит документацию.

In [ ]:
import sys
import fastapi, pydantic, starlette

print("Python   ", sys.version.split()[0])
print("FastAPI  ", fastapi.__version__)
print("Pydantic ", pydantic.VERSION)
print("Starlette", starlette.__version__)

## 2. Первое приложение и запуск

Самое маленькое приложение FastAPI:

```python
# main.py
from fastapi import FastAPI

app = FastAPI(title="Покрытие")


@app.get("/api/hello")
def hello():
    return {"message": "Привет, водитель!"}
```

Запуск из терминала в папке с файлом:

```
fastapi dev main.py
```

Команда `fastapi dev` находит в файле объект `app`, запускает сервер Uvicorn на адресе `http://127.0.0.1:8000` и следит за файлами: после сохранения кода сервер перезапускается сам. Это режим разработки. Для настоящей работы сервиса используется `fastapi run`: без перезапуска и с доступом из сети. Если команда `fastapi` не найдена (в Windows так бывает, когда папка Scripts не прописана в путях поиска программ), запускайте `python -m fastapi dev main.py`.

Что произошло в коде:
- `app = FastAPI(...)` – объект приложения. В нем хранится таблица маршрутов;
- `@app.get("/api/hello")` – декоратор с параметром: записывает функцию `hello` в таблицу маршрутов для метода GET и пути `/api/hello`;
- функция возвращает словарь, а FastAPI сам превращает его в JSON, ставит заголовки `Content-Type: application/json` и `Content-Length`.

В ноутбуке сервер не нужен: тестовый клиент `TestClient` передает запрос приложению прямо в памяти и возвращает ответ с теми же полями, что у библиотеки `requests`.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Покрытие")


@app.get("/api/hello")
def hello():
    return {"message": "Привет, водитель!"}


client = TestClient(app)
r = client.get("/api/hello")
print("код:          ", r.status_code)
print("Content-Type: ", r.headers["content-type"])
print("Content-Length:", r.headers["content-length"])
print("тело:         ", r.text)
print("как словарь:  ", r.json())

Обратите внимание на `Content-Length`. В Т4.Б1.1 вы сами считали длину тела в байтах, потому что русская буква в UTF-8 занимает 2 байта. Теперь это делает FastAPI.

Ответы 404 и 405 фреймворк тоже отправляет сам:

In [ ]:
r = client.get("/api/nowhere")
print("GET /api/nowhere  ->", r.status_code, r.json())

r = client.post("/api/hello")
print("POST /api/hello   ->", r.status_code, r.json(), "  Allow:", r.headers["allow"])

**Страницы HTML и перенаправление.** По умолчанию FastAPI отвечает в формате JSON. Чтобы отдать страницу, указывают класс ответа `HTMLResponse`. Перенаправление – объект `RedirectResponse`. Над одной функцией можно поставить несколько декораторов, и она будет отвечать по нескольким адресам. Параметр `include_in_schema=False` убирает маршрут из документации: документация описывает программный интерфейс для программ, а страницы предназначены людям.

In [ ]:
from fastapi.responses import HTMLResponse, RedirectResponse


@app.get("/", response_class=HTMLResponse, include_in_schema=False)
def index_page():
    return "<!doctype html><html lang='ru'><title>Покрытие</title><h1>Сервис «Покрытие»</h1></html>"


@app.get("/index.html", include_in_schema=False)
@app.get("/home", include_in_schema=False)
def old_address():
    return RedirectResponse("/", status_code=301)


r = client.get("/")
print("GET /      ->", r.status_code, r.headers["content-type"])
r = client.get("/home", follow_redirects=False)
print("GET /home  ->", r.status_code, "Location:", r.headers["location"])
r = client.get("/home")
print("с переходом по перенаправлению ->", r.status_code, r.url)

**Файлы.** В Т4.Б1.1 отдача файлов с проверкой пути была заданием повышенной сложности. Здесь это одна строка: `app.mount("/static", StaticFiles(directory="static"), name="static")`. Объект `StaticFiles` сам выбирает `Content-Type` по расширению и не выпускает за пределы папки.

## 3. Маршруты и методы, параметры пути и запроса

Методу HTTP соответствует свой декоратор: `@app.get`, `@app.post`, `@app.put`, `@app.delete`. Данные запроса FastAPI раскладывает по аргументам функции:

| Откуда | Пример | Как объявить |
| --- | --- | --- |
| часть пути | `/api/trips/7` | `@app.get("/api/trips/{trip_id}")` и аргумент `trip_id: int` |
| параметр запроса после `?` | `/api/braking?v=60&mu=0.7` | аргумент `v: float`, имени которого нет в пути |
| тело запроса в формате JSON | `{"date": "2026-11-05", ...}` | аргумент с типом модели Pydantic (раздел 6) |

**Подсказки типов** – запись `имя: тип` у параметров и `-> тип` у результата. Сам Python их не проверяет. FastAPI читает их один раз при запуске приложения и по ним:
1. превращает строку из адреса в нужный тип (`"7"` в 7, `"0.7"` в 0.7);
2. отвечает кодом 422, если превратить нельзя;
3. описывает параметр в документации.

In [ ]:
app = FastAPI(title="Покрытие")
TRIPS = {7: "поездка по снегу", 8: "поездка по грунту"}


@app.get("/api/trips/{trip_id}")
def get_trip(trip_id: int):
    print("   функция получила", repr(trip_id), type(trip_id).__name__)
    return {"id": trip_id, "title": TRIPS.get(trip_id)}


@app.get("/api/convert")
def convert(v: float, unit: str = "m/s"):
    return {"v_kmh": v, "result": round(v / 3.6, 2), "unit": unit}


client = TestClient(app)
print("GET /api/trips/7 ->", client.get("/api/trips/7").json())
print("GET /api/convert?v=90 ->", client.get("/api/convert", params={"v": 90}).json())
print("GET /api/convert?v=90&unit=км/ч ->", client.get("/api/convert", params={"v": 90, "unit": "км/ч"}).json())

Функция получила число 7, а не строку `"7"`: преобразование сделал FastAPI. У параметра `unit` есть значение по умолчанию, поэтому он необязательный. Параметр `v` без значения по умолчанию обязателен.

Что будет, если номер поездки не число:

In [ ]:
r = client.get("/api/trips/abc")
print(r.status_code)
print(r.json())

Функция `get_trip` даже не вызвалась: в выводе нет строки «функция получила». FastAPI проверил данные до вызова и сам ответил кодом **422 Unprocessable Content** («запрос понятен, но данные в нем неверные»).

Ответ на ошибку проверки всегда устроен одинаково: список `detail`, в котором у каждой ошибки есть
- `loc` – место: откуда параметр (`path` – путь, `query` – параметры запроса, `body` – тело) и его имя;
- `type` – вид ошибки (`int_parsing` – не удалось прочитать целое число);
- `msg` – текст для человека;
- `input` – что пришло.

Программа-клиент по `loc` может показать ошибку рядом с нужным полем формы. Тексты ошибок на английском. Единый формат ошибок на русском языке настроим в Т4.Б1.3.

## 4. Проверка параметров: `Query`, `Path` и ответ 422

Тип `float` пропустит и отрицательную скорость. Ограничения задаются так:

```python
from typing import Annotated
from fastapi import Query

def braking(v: Annotated[float, Query(gt=0, le=200, description="Скорость, км/ч")]): ...
```

`Annotated[тип, сведения]` – стандартная запись Python: «значение типа `float`, и вот дополнительные сведения о нем». Python сведения не использует, а FastAPI читает. `Query(...)` говорит, что значение берется из параметров запроса, и задает ограничения:

| Ограничение | Смысл | Пример |
| --- | --- | --- |
| `gt` | больше (greater than) | `gt=0`: 0 нельзя |
| `ge` | больше или равно (greater or equal) | `ge=1`: 1 можно |
| `lt` | меньше (less than) | `lt=100` |
| `le` | меньше или равно (less or equal) | `le=200` |
| `min_length`, `max_length` | длина строки | `max_length=200` |
| `description` | описание для документации | `description="Скорость, км/ч"` |

Для части пути то же самое делает `Path(...)`, например `trip_id: Annotated[int, Path(ge=1)]`.

Калькулятор тормозного пути из Т4.Б1.1 на FastAPI:

In [ ]:
from typing import Annotated
from fastapi import Query

app = FastAPI(title="Покрытие")
calls = 0


@app.get("/api/braking")
def braking(
    v: Annotated[float, Query(gt=0, le=200, description="Скорость, км/ч")],
    mu: Annotated[float, Query(gt=0, le=1.2, description="Коэффициент сцепления")],
    t: Annotated[float, Query(gt=0, le=3, description="Время реакции, с")] = 1.0,
):
    global calls
    calls += 1
    return {"v": v, "mu": mu, "t": t, "braking_distance": round(v / 3.6 * t + v ** 2 / (254 * mu), 1)}


client = TestClient(app)
print(client.get("/api/braking", params={"v": 60, "mu": 0.7}).json())
print(client.get("/api/braking", params={"v": 60, "mu": 0.7, "t": 0.3}).json())

In [ ]:
def show_errors(params):
    r = client.get("/api/braking", params=params)
    print(params, "->", r.status_code)
    for e in r.json()["detail"]:
        print(f"   {'.'.join(map(str, e['loc'])):8} {e['type']:16} {e['msg']}")


calls_before = calls
show_errors({"v": -10, "mu": 0.7})          # отрицательная скорость
show_errors({"v": 60, "mu": 0})             # коэффициент 0
show_errors({"v": 60, "mu": "0,7"})         # десятичная запятая
show_errors({"v": "abc", "mu": 5})          # две ошибки сразу
show_errors({})                             # нет обязательных параметров
print("функция braking вызывалась за это время", calls - calls_before, "раз")

Что видно из опыта:
1. FastAPI перечисляет **все** ошибки сразу, а не только первую.
2. Десятичная запятая не проходит. В Т4.Б1.1 наша функция `read_number` принимала `0,7`. Программный интерфейс работает строже: формат обмена данными один для всех, а запятую при вводе заменяет сама страница или мобильное приложение.
3. При ошибках функция `braking` не вызывалась ни разу. Внутри функции данные уже проверены, поэтому проверять их еще раз не нужно.

Это ответ на главный вопрос занятия: FastAPI знает, что скорость должна быть положительным числом, из подписи функции. Подсказка `float` задает тип, `Query(gt=0, le=200)` задает границы.

## 5. Модели Pydantic

Поездка состоит из нескольких полей: дата, расстояние, покрытие, комментарий. Описывать каждое параметром функции неудобно. Для составных данных используют **модели Pydantic**: класс-наследник `BaseModel`, у которого каждое поле – подсказка типа с ограничениями.

```python
class TripIn(BaseModel):
    date: datetime.date
    distance: float = Field(gt=0, le=2000, description="Расстояние, км")
    surface: Surface
    note: str = Field(default="", max_length=200)
```

- `Field(...)` – ограничения и описание поля, как `Query(...)` у параметров;
- поле со значением по умолчанию необязательное (`note`);
- тип `datetime.date` проверяет, что дата записана в виде ГГГГ-ММ-ДД и существует в календаре;
- тип `Surface` – **перечисление** (`Enum`): тип с заранее заданным списком значений. `StrEnum` – перечисление, значения которого строки.

Модели Pydantic работают и без FastAPI. Проверим модель напрямую:

In [ ]:
import datetime
from enum import StrEnum
from pydantic import BaseModel, Field, ValidationError


class Surface(StrEnum):
    dry = "dry"
    wet = "wet"
    dirt = "dirt"
    snow = "snow"


class TripIn(BaseModel):
    date: datetime.date = Field(description="Дата поездки в виде ГГГГ-ММ-ДД")
    distance: float = Field(gt=0, le=2000, description="Расстояние, км")
    surface: Surface = Field(description="Покрытие")
    note: str = Field(default="", max_length=200, description="Комментарий водителя")


trip = TripIn.model_validate({"date": "2026-11-05", "distance": "12.5", "surface": "wet"})
print(trip)
print("тип даты:     ", type(trip.date).__name__)
print("тип расстояния:", type(trip.distance).__name__)
print("покрытие:     ", repr(trip.surface))
print("словарь:      ", trip.model_dump())

Pydantic не только проверил данные, но и **преобразовал** их: строка `"2026-11-05"` стала объектом даты, строка `"12.5"` – числом, `"wet"` – значением перечисления. Метод `model_dump()` превращает модель обратно в словарь.

Ошибки Pydantic собирает все сразу, с местом и видом каждой:

In [ ]:
bad = {"date": "2026-02-30", "distance": -5, "surface": "lava", "note": "а" * 201}
try:
    TripIn.model_validate(bad)
except ValidationError as error:
    print("ошибок:", error.error_count())
    for e in error.errors():
        print(f"   {'.'.join(map(str, e['loc'])):10} {e['type']:28} {e['msg']}")

Такую проверку в Т4.Б1.1 делала функция `validate_trip` на 20 строк. Здесь она описана четырьмя строками модели и, главное, описана один раз: по той же модели FastAPI проверит запрос и построит документацию.

**Схема модели.** По модели Pydantic строит описание в формате **JSON Schema** – стандарте описания структуры данных JSON. Именно его FastAPI вставляет в документацию:

In [ ]:
import json

schema = TripIn.model_json_schema()
print(json.dumps(schema["properties"]["distance"], ensure_ascii=False, indent=2))
print("обязательные поля:", schema["required"])
print("значения Surface:", schema["$defs"]["Surface"]["enum"])

Ограничение `gt=0` записано как `exclusiveMinimum: 0` (строгий минимум), `le=2000` – как `maximum: 2000`. Поля `note` нет среди обязательных: у него значение по умолчанию.

## 6. Тело запроса и код 201

Если у аргумента функции тип модели Pydantic, FastAPI берет его из **тела запроса** в формате JSON. Последовательность такая:
1. прочитать тело запроса;
2. разобрать JSON в словарь (неверный JSON – ответ 422);
3. проверить словарь по модели (ошибки – ответ 422 со всеми ошибками);
4. создать объект модели и вызвать функцию.

Новая запись создается методом POST. Успешный ответ на создание – код **201 Created**: «создан новый ресурс». Его задают в декораторе: `status_code=201`.

Аргумент с типом `Request` – сам запрос целиком. Из него можно узнать, например, адрес клиента `request.client.host`.

In [ ]:
from fastapi import Request

app = FastAPI(title="Покрытие")
TRIPS = []


@app.post("/api/trips", status_code=201)
def create_trip(trip: TripIn, request: Request):
    record = trip.model_dump()
    record["id"] = len(TRIPS) + 1
    record["created_at"] = datetime.datetime(2026, 11, 5, 14, 30)
    record["client_host"] = request.client.host          # служебное поле
    TRIPS.append(record)
    return record


client = TestClient(app)
r = client.post("/api/trips", json={"date": "2026-11-05", "distance": 12.5, "surface": "wet", "note": "дождь"})
print(r.status_code, r.json())

r = client.post("/api/trips", json={})
print(r.status_code, [(e["loc"], e["type"]) for e in r.json()["detail"]])
print("сохранено поездок:", len(TRIPS))

Тестовый клиент представляется адресом `testclient`. Настоящий браузер на том же компьютере пришел бы с адреса `127.0.0.1`.

Посмотрите на первый ответ: в нем есть поле `client_host`. Это служебное поле: адрес компьютера водителя клиенту не нужен, а раздавать его каждому, кто прислал запрос, нельзя. Это исправляет модель ответа.

## 7. Модель ответа `response_model`

Модель ответа – обещание клиенту: какие поля сервер вернет. Ее указывают в декораторе: `response_model=TripOut`. Функция может вернуть словарь с лишними полями, а FastAPI перед отправкой:
1. оставит только поля модели ответа;
2. проверит, что обещанные поля на месте (если нет – это ошибка сервера, код 500);
3. опишет ответ в документации.

Зачем это нужно:
- **безопасность**: служебные данные (адрес клиента, хеш пароля, внутренние пометки) не уходят наружу, даже если функция вернула их по ошибке;
- **постоянство**: клиенты (страница на React, мобильное приложение) получают одинаковый ответ, даже когда внутреннее устройство записи меняется;
- **документация**: клиент заранее знает, что придет в ответе.

In [ ]:
from fastapi import HTTPException, Path


class TripOut(BaseModel):
    id: int
    date: datetime.date
    distance: float
    surface: Surface
    note: str
    created_at: datetime.datetime


app = FastAPI(title="Покрытие")
TRIPS = []


@app.post("/api/trips", status_code=201, response_model=TripOut)
def create_trip(trip: TripIn, request: Request):
    record = trip.model_dump()
    record["id"] = len(TRIPS) + 1
    record["created_at"] = datetime.datetime(2026, 11, 5, 14, 30)
    record["client_host"] = request.client.host
    TRIPS.append(record)
    return record                       # возвращаем всю запись


@app.get("/api/trips/{trip_id}", response_model=TripOut)
def get_trip(trip_id: Annotated[int, Path(ge=1)]):
    for record in TRIPS:
        if record["id"] == trip_id:
            return record
    raise HTTPException(status_code=404, detail=f"Поездки с номером {trip_id} нет")


client = TestClient(app)
r = client.post("/api/trips", json={"date": "2026-11-05", "distance": 12.5, "surface": "wet"})
print("POST ->", r.status_code, r.json())
print("в списке TRIPS:", TRIPS[0])
print("GET /api/trips/1  ->", client.get("/api/trips/1").status_code)
print("GET /api/trips/99 ->", client.get("/api/trips/99").status_code, client.get("/api/trips/99").json())
print("GET /api/trips/0  ->", client.get("/api/trips/0").status_code)

В списке `TRIPS` поле `client_host` есть, а в ответе его нет: отфильтровала модель `TripOut`.

**Ответ 404 своими руками.** Если запись не найдена, функция вызывает исключение `HTTPException(status_code=404, detail="...")`. FastAPI перехватывает его и отправляет ответ с этим кодом и текстом. Сравните два ответа:
- `/api/trips/0` – 422: запрос неверный (номер меньше 1), ответил FastAPI до вызова функции;
- `/api/trips/99` – 404: запрос верный, но такой поездки нет, решила наша функция.

## 8. Автоматическая документация

По маршрутам, подсказкам типов и моделям FastAPI строит описание программного интерфейса в формате **OpenAPI** – стандарте описания интерфейсов, которые обмениваются данными по HTTP. Описание доступно по трем адресам:

| Адрес | Что там |
| --- | --- |
| `/openapi.json` | описание в формате OpenAPI: файл JSON для программ |
| `/docs` | страница **Swagger UI**: документация с кнопкой «Try it out» для выполнения запроса прямо со страницы |
| `/redoc` | страница **ReDoc**: та же документация в виде справочника для чтения |

Документацию читают: разработчик клиентской части на React (Блок 2), разработчик мобильного приложения на Kotlin (Тема 5), программы, которые по `/openapi.json` создают код клиента автоматически, и сам автор через месяц.

Чтобы документация была понятна человеку, добавляют:
- описание приложения: `FastAPI(title=..., version=..., description=...)`;
- **теги** – группы маршрутов на странице `/docs`: `tags=["Поездки"]` у маршрута и описания тегов в `FastAPI(openapi_tags=[...])`;
- краткое описание маршрута `summary="Добавить поездку"`; подробное описание FastAPI берет из строки документации функции (docstring);
- описания параметров и полей: `description="..."` в `Query`, `Path`, `Field`;
- пример тела запроса в модели: `model_config = ConfigDict(json_schema_extra={"examples": [...]})`;
- описания ответов с ошибками: `responses={404: {"description": "..."}}`.

In [ ]:
from pydantic import ConfigDict


class TripIn(BaseModel):
    "Поездка, которую присылает клиент."
    model_config = ConfigDict(json_schema_extra={
        "examples": [{"date": "2026-11-05", "distance": 12.5, "surface": "wet", "note": "дождь после обеда"}],
    })
    date: datetime.date = Field(description="Дата поездки в виде ГГГГ-ММ-ДД")
    distance: float = Field(gt=0, le=2000, description="Расстояние, км")
    surface: Surface = Field(description="Покрытие: dry, wet, dirt или snow")
    note: str = Field(default="", max_length=200, description="Комментарий водителя")


app = FastAPI(
    title="Покрытие",
    version="0.2.0",
    description="Учебный сервис беспилотного автомобиля",
    openapi_tags=[{"name": "Поездки", "description": "Журнал поездок"}],
)


@app.post("/api/trips", status_code=201, response_model=TripOut, tags=["Поездки"], summary="Добавить поездку")
def create_trip(trip: TripIn):
    "Принимает поездку в формате JSON, проверяет ее и сохраняет."
    return {**trip.model_dump(), "id": 1, "created_at": datetime.datetime(2026, 11, 5, 14, 30)}


spec = TestClient(app).get("/openapi.json").json()
print("info:", spec["info"])
op = spec["paths"]["/api/trips"]["post"]
print("summary:    ", op["summary"])
print("description:", op["description"])
print("tags:       ", op["tags"])
print("ответы:     ", list(op["responses"]))
print("пример тела:", spec["components"]["schemas"]["TripIn"]["examples"][0])

В ответах документации есть не только 201, но и 422: FastAPI знает, что при ошибке проверки ответит именно так, и описывает это сам.

**Как пользоваться страницей `/docs`** (запустите `fastapi dev main.py` в папке практической работы):
1. Раскройте маршрут, например `POST /api/trips`.
2. Нажмите «Try it out»: поле тела запроса станет редактируемым, в нем уже стоит пример из модели.
3. Измените данные и нажмите «Execute».
4. Ниже появятся команда `curl` для того же запроса, адрес, код ответа, тело и заголовки ответа.

Попробуйте отправить `"distance": -5` и прочитайте ответ 422 по полям `loc`, `type`, `msg`.

## 9. Функции `def` и `async def`

В документации FastAPI часто встречаются обработчики `async def`. В курсе мы пишем обычные `def`. Разберемся почему.

**Асинхронная функция** (`async def`) может приостановиться в точке `await`, пока ждет чего-то долгого (ответа сети, таймера), и отдать управление другим задачам. Все асинхронные функции выполняет **цикл событий** – один поток, который переключается между задачами только в точках `await`.

FastAPI выполняет обработчики так:
- `async def` – прямо в цикле событий. Если внутри вызвать что-то долгое **без** `await` (например, `time.sleep` или расчет модели), цикл событий остановится целиком, и все остальные запросы будут ждать;
- `def` – в отдельном потоке из запаса потоков (по умолчанию до 40 одновременно). Долгая работа в одном потоке не мешает остальным.

Проверим на простом опыте без сервера: три задачи по 0,5 с в цикле событий.

In [ ]:
import asyncio
import time


async def with_time_sleep(name):
    time.sleep(0.5)                 # обычная пауза: останавливает весь цикл событий
    return name


async def with_await(name):
    await asyncio.sleep(0.5)        # асинхронная пауза: цикл событий занимается другими задачами
    return name


async def run_three(task):
    started = time.perf_counter()
    await asyncio.gather(task("A"), task("B"), task("C"))
    return time.perf_counter() - started


def run(coro):
    # В Jupyter цикл событий уже запущен, поэтому опыт выполняется в отдельном потоке со своим циклом.
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(asyncio.run, coro).result()


print(f"async def + time.sleep:          {run(run_three(with_time_sleep)):.2f} с")
print(f"async def + await asyncio.sleep: {run(run_three(with_await)):.2f} с")

С `time.sleep` задачи идут по очереди (около 1,5 с), с `await asyncio.sleep` – одновременно (около 0,5 с).

Почему в курсе `def`:
1. Модель scikit-learn, драйвер базы данных psycopg 3 в синхронном режиме и большинство библиотек не умеют `await`. Внутри `async def` они останавливали бы весь сервер.
2. Функцию `def` FastAPI сам запускает в отдельном потоке, поэтому она безопасна по умолчанию.
3. Ошибку «долгий вызов без `await` внутри `async def`» трудно заметить: на одном запросе все работает, а под нагрузкой сервер замирает.

Тот же опыт на настоящем сервере с пятью одновременными запросами – задание повышенной сложности практической работы. Асинхронный вариант серверной части разбирается в Т4.Б3.2.

Единственное место, где в этом занятии нужен `async def`, – **промежуточный обработчик** (middleware): функция, через которую проходит каждый запрос до маршрута и каждый ответ после него. Она должна дождаться ответа через `await call_next(request)`.

## 10. Итоги

| Что нужно | Как в FastAPI |
| --- | --- |
| маршрут | декоратор `@app.get("/путь")`, `@app.post(...)` |
| страница HTML | `response_class=HTMLResponse`, `include_in_schema=False` |
| перенаправление | `RedirectResponse("/", status_code=301)` |
| файлы | `app.mount("/static", StaticFiles(directory=...))` |
| параметр пути | `/api/trips/{trip_id}` и `trip_id: Annotated[int, Path(ge=1)]` |
| параметр запроса | `v: Annotated[float, Query(gt=0, le=200)]` |
| тело запроса | аргумент с типом модели Pydantic |
| модель данных | `class TripIn(BaseModel)`, поля с `Field(...)`, перечисление `StrEnum` |
| код ответа | `status_code=201` в декораторе |
| модель ответа | `response_model=TripOut` |
| ошибка «не найдено» | `raise HTTPException(status_code=404, detail="...")` |
| документация | `/docs`, `/redoc`, `/openapi.json`; `summary`, `tags`, `description`, примеры |
| запуск | `fastapi dev main.py`, адрес `http://127.0.0.1:8000` |
| тесты | `TestClient(app)` |

**Словарь**

| Термин | Значение |
| --- | --- |
| фреймворк | библиотека, которая задает каркас программы и сама вызывает наш код |
| ASGI | Asynchronous Server Gateway Interface: стандарт связи веб-сервера и приложения на Python |
| Uvicorn | сервер ASGI, который принимает соединения и передает запросы приложению FastAPI |
| маршрут | пара «метод и путь» и функция, которая отвечает на такой запрос |
| параметр пути | часть пути в фигурных скобках: `{trip_id}` |
| параметр запроса | пара `имя=значение` после знака `?` в адресе |
| подсказка типа | запись `имя: тип`; Python ее не проверяет, FastAPI по ней проверяет и преобразует данные |
| модель Pydantic | класс-наследник `BaseModel`: поля с типами и ограничениями |
| перечисление | тип с заранее заданным списком значений (`Enum`, `StrEnum`) |
| 422 Unprocessable Content | ответ «данные запроса неверные» со списком ошибок `detail` |
| 201 Created | ответ «создан новый ресурс» |
| модель ответа | модель, по которой FastAPI фильтрует и проверяет ответ |
| OpenAPI | стандарт описания программных интерфейсов, работающих по HTTP |
| Swagger UI, ReDoc | страницы документации по описанию OpenAPI (`/docs` и `/redoc`) |
| JSON Schema | стандарт описания структуры данных JSON |
| цикл событий | поток, который выполняет асинхронные функции и переключается между ними в точках `await` |
| промежуточный обработчик | функция, через которую проходит каждый запрос и каждый ответ (middleware) |

**Вопросы для самопроверки**

1. Какие функции сервера из Т4.Б1.1 не нужны в FastAPI и кто теперь делает их работу?
2. Откуда FastAPI знает, что параметр `v` берется из адреса после `?`, а `trip_id` – из пути?
3. Что означают `gt`, `ge`, `lt`, `le`? Как записать «от 1 до 10 включительно»?
4. Чем ответ 422 отличается от ответа 404? Кто отправляет каждый из них?
5. Какие четыре поля есть у каждой ошибки в ответе 422 и что в них?
6. Зачем модель ответа, если функция и так возвращает нужные данные?
7. Где в документации видно, что поле `distance` должно быть больше 0?
8. Почему внутри `async def` нельзя вызывать `time.sleep`?

**Задачи**

1. Объявите маршрут `GET /api/convert` с параметром `kmh` (больше или равно 0, не больше 300) и ответом `{"kmh": ..., "ms": ...}` – скорость в метрах в секунду с округлением до 0,01. Проверьте его тестовым клиентом на значениях 90, 0, -1, `abc`.
2. Опишите модель `Car`: номер (строка от 6 до 9 символов), марка (строка), год выпуска (целое от 1990 до текущего года), тип шин (перечисление `summer`, `winter`). Проверьте ее на трех правильных и трех неправильных словарях и выведите ошибки.
3. Добавьте к модели `TripIn` необязательное поле `driver` (строка или `None`, по умолчанию `None`). Как оно выглядит в схеме `model_json_schema()`?
4. Напишите маршрут `GET /api/surfaces/{code}`, где `code` – значение перечисления `Surface`. Что вернет FastAPI на `/api/surfaces/lava`? Как выглядит этот параметр на странице `/docs`?
5. Какие поля модели `User` (имя, электронная почта, хеш пароля, дата регистрации, признак администратора) вы включите в модель ответа для списка пользователей, а какие нет? Объясните выбор.